# Practica de Programacion y Algoritmia Avanzada – GIA

El objetivo de esta actividad es poner en practica los conocimientos de programacion dinamica, para tal caso, se os propone implementar el algoritmo determinista Cocke–Kasami–Younger.

El algoritmo CKY determinista es un algoritmo utilizado para analizar la estructura sintactica de cadenas de texto, y determinar si pueden ser derivadas apartir de las reglas de una gramatica.

In [ ]:
# INPUT
# Gramar = {} # CNF grammar
# Word = []

# terminals = set()
# non_terminals = set()

In [11]:
# MAIN PROGRAM
def input_grammar(input_grammar):
    # Read grammar from input
    grammar = {}
    while True:
        line = input().strip()
        if line == "":
            break
        lhs, rhs = line.split("->")
        lhs = lhs.strip()
        rhs = rhs.strip().split("|")
        grammar[lhs] = [r.strip() for r in rhs]
    return grammar
    
def check_grammar(grammar):
    # Check if the grammar is in CNF
    for lhs, rhs_list in grammar.items():
        for rhs in rhs_list:
            if len(rhs) > 2 or (len(rhs) == 2 and not all(c.isupper() for c in rhs)):
                print(f"Grammar is not in CNF: {lhs} -> {rhs}")
                return False
    print("Grammar is in CNF")
    
def check_word(word, grammar):
    # Check if the word can be generated by the grammar
    def generate(lhs, word):
        if not word:
            return lhs == ""
        if lhs not in grammar:
            return False
        for rhs in grammar[lhs]:
            if len(rhs) == 1 and rhs[0] == word[0]:
                if generate(rhs[0], word[1:]):
                    return True
            elif len(rhs) == 2 and rhs[0] == word[0] and rhs[1] == word[1]:
                if generate(rhs[1], word[2:]):
                    return True
        return False
    
def test(word, grammar):
    # Test if the word can be generated by the grammar:
    # input_grammar(input_grammar)
    # grammar = input_grammar(input_grammar)
    if not check_grammar(grammar):
        return False
    else:
        return check_word(word, grammar)     


In [30]:
def cargar_gramatica(ruta_archivo):
    gramaticas={}
    reglas = {}
    palabras = []
    n = 1

    with open(ruta_archivo) as f:
        for linea in f:
            if linea:
                linea = linea.strip()

                #si hay \n es q hay nueva gramatica, y se guarda lo que hemos leido
                if linea == "" or linea == " ":
                    if reglas and palabras:
                        gramaticas[f"G{n}"] = {
                            "reglas": reglas,
                            "palabras": palabras
                        }
                        n += 1
                        reglas = {}
                        palabras = []

                if '->' in linea:
                    l_iz, l_der = linea.split('->')
                    l_iz = l_iz.strip()
                    alternativas = l_der.split('|')

                    for alt in alternativas:
                        alt = alt.strip()
                        if alt in ['epsilon', 'ε']:
                            produccion = ['epsilon']  # Representamos epsilon siempre como 'epsilon'
                        else:
                            produccion = alt.split() if ' ' in alt else list(alt)
                            reglas.setdefault(l_iz, []).append(produccion)

                            
                else:
                    if linea != "":
                        palabras.append(linea)
            

    # Si input no acaba en /n añadir lo ultimo leido               
    if reglas and palabras:
        gramaticas[f"G{n}"] = {
            "reglas": reglas,
            "palabras": palabras
        }

    return gramaticas


In [31]:
gramatica= cargar_gramatica('c:/Users/veron/Documents/UNI/2º/Febrero/PAA/Practica/input.txt')
print(gramatica)

{'G1': {'reglas': {'S': [['a'], ['X', 'A'], ['A', 'X'], ['b']], 'C': [['A', 'B', 'F']], 'D': [['a', 'A']], 'E': [['B', 'b']], 'F': [['t']], 'A': [['R', 'B']], 'B': [['A', 'X'], ['b'], ['a']], 'X': [['a']], 'R': [['X', 'B']]}, 'palabras': ['aabb', 'abba', 'aaabbba']}, 'G2': {'reglas': {'S': [['a'], ['X', 'A'], ['A', 'X'], ['b']], 'C': [['A', 'B', 'F']], 'D': [['a', 'A']], 'E': [['B', 'b']]}, 'palabras': ['efw', 'asddb', 'asddb', 'a']}}


In [27]:
def convertir_cnf(gramatica):

    simbolo_inicial = list(gramatica.keys())[0]

    # No terminales: claves del diccionario
    no_terminales = set(gramatica.keys())
    terminales = set()

    # Recorrer todas las reglas para encontrar terminales
    for reglas in gramatica.values():
        for produccion in reglas:
            for simbolo in produccion:
                if simbolo not in no_terminales and simbolo != 'epsilon':
                    terminales.add(simbolo)

    nueva_gramatica = {}
    contador = 0

    # Paso 1: Nuevo símbolo inicial si hace falta
    if any(simbolo_inicial in prod for reglas in gramatica.values() for prod in reglas):
        nueva_gramatica['S0'] = [[simbolo_inicial]]
        simbolo_inicial = 'S0'

    # Copiar las reglas originales
    for no_terminal, reglas in gramatica.items():
        nueva_gramatica[no_terminal] = reglas.copy()

    # Paso 2: Eliminar epsilon
    vacios = set()
    for nt, reglas in nueva_gramatica.items():
        for regla in reglas:
            if regla == ['epsilon']:
                vacios.add(nt)
    for nt in vacios:
        nueva_gramatica[nt] = [r for r in nueva_gramatica[nt] if r != ['epsilon']]
        for nt2, reglas in nueva_gramatica.items():
            nuevas = []
            for regla in reglas:
                if nt in regla:
                    nueva = [s for s in regla if s != nt]
                    if nueva and nueva != regla and nueva not in reglas:
                        nuevas.append(nueva)
            nueva_gramatica[nt2].extend(nuevas)

    # Paso 3: Eliminar producciones unitarias
    for nt in list(nueva_gramatica):
        nuevas = []
        for regla in nueva_gramatica[nt]:
            if len(regla) == 1 and regla[0] in nueva_gramatica and regla[0] != nt:
                nuevas.extend([r for r in nueva_gramatica[regla[0]] if r not in nueva_gramatica[nt]])
        nueva_gramatica[nt].extend(nuevas)  

    # Paso 4: Dividir producciones largas
    for nt in list(nueva_gramatica):
        nuevas = []
        for regla in nueva_gramatica[nt]:
            while len(regla) > 2:
                nuevo = f"X{contador}"
                contador += 1
                nueva_gramatica[nuevo] = [[regla[0], regla[1]]]
                regla = [nuevo] + regla[2:]
            nuevas.append(regla)
        nueva_gramatica[nt] = nuevas

    # Paso 5: Separar terminales si hay más de uno en la producción
    reemplazos = {}
    for nt in list(nueva_gramatica):
        for i, regla in enumerate(nueva_gramatica[nt]):
            if len(regla) == 2:
                for j in range(2):
                    simbolo = regla[j]
                    if simbolo in terminales:
                        if simbolo not in reemplazos:
                            nuevo = f"T{contador}"
                            contador += 1
                            nueva_gramatica[nuevo] = [[simbolo]]
                            reemplazos[simbolo] = nuevo
                        regla[j] = reemplazos[simbolo]

    return nueva_gramatica


In [5]:
def es_cnf(gramatica):
    simbolo_inicial = list(gramatica.keys())[0]
    no_terminales = set(gramatica.keys())
    terminales = set()

    
    # Recorremos toda la gramática para ver qué símbolos podrían ser terminales
    for reglas in gramatica.values():
        for produccion in reglas:
            for simbolo in produccion:
                if simbolo == simbolo_inicial or simbolo == 'epsilon' :
                    return False
                if simbolo not in no_terminales :
                    terminales.add(simbolo)

    # Ahora comprobamos si cada producción cumple las reglas de la CNF
    for lado_izq, reglas in gramatica.items():
        for produccion in reglas:
            
            if len(produccion) == 1:
                # Caso A -> a
                if produccion[0] not in terminales:
                    return False
                    
            elif len(produccion) == 2:
                # Caso A -> B C
                if produccion[0] in terminales or produccion[1] in terminales:
                    return False
                    
            else:
                # Producción con más de 2 símbolos ej A-> BCD
                return False
                
    
     
    return True    
    


In [12]:
gramatica= cargar_gramatica('c:/Users/veron/Documents/UNI/2º/Febrero/PAA/Practica/input.txt')
print(gramatica)


{'G1': {'reglas': {'S': [['a'], ['XA'], ['AX'], ['b'], ['epsilon'], ['a'], ['XA'], ['AX'], ['b'], ['epsilon']], 'C': [['ABF'], ['ABF']], 'D': [['aA'], ['aA']], 'E': [['Bb'], ['Bb']], 'F': [['t']], 'A': [['RB']], 'B': [['AX'], ['b'], ['a']], 'X': [['a']], 'R': [['XB']]}, 'palabras': ['aabb', 'abba', 'aaabbba', '', 'efw', 'asddb', 'asddb', 'a']}}


In [31]:
es_cnf(gramatica)

False

In [32]:
if not es_cnf(gramatica):
    print('No esta en CNF, convirtiendo...')
    gramatica= convertir_cnf(gramatica)
    print(es_cnf(gramatica))
    print(gramatica)


No esta en CNF, convirtiendo...
True
{'S': [['a'], ['X', 'A'], ['A', 'X'], ['b']], 'C': [['X0', 'F']], 'D': [['T1', 'A']], 'E': [['B', 'T2']], 'F': [['T']], 'A': [['R', 'B']], 'B': [['A', 'X'], ['b'], ['a']], 'X': [['a']], 'R': [['X', 'B']], 'X0': [['A', 'B']], 'T1': [['a']], 'T2': [['b']]}


In [78]:
def cky(gramatica, palabra):
    n = len(palabra)
    tabla = [[set() for _ in range(n)] for _ in range(n)]

    # Crear un diccionario inverso: RHS -> LHS
    inversa = {}
    for lhs, producciones in gramatica.items():
        for rhs in producciones:
            clave = tuple(rhs)
            inversa.setdefault(clave, []).append(lhs)

    # Rellenar la diagonal (longitud 1): A -> a
    for i, simbolo in enumerate(palabra):
        if (simbolo,) in inversa:
            tabla[i][i].update(inversa[(simbolo,)])

    # Rellenar el resto de la tabla
    for longitud in range(2, n + 1):  # tamaño del fragmento
        for i in range(n - longitud + 1):
            j = i + longitud - 1
            for k in range(i, j):
                izquierda = tabla[i][k]
                derecha = tabla[k + 1][j]
                for B in izquierda:
                    for C in derecha:
                        if (B, C) in inversa:
                            tabla[i][j].update(inversa[(B, C)])
    print(tabla)
    # Verificamos si el símbolo inicial S genera toda la cadena
    return 'S' in tabla[0][n - 1]


In [79]:
Gramar1 = {
    'S': [['a'], ['X', 'A'], ['A', 'X'], ['b']],
    'A': [['R', 'B']],
    'B': [['A', 'X'], ['b'], ['a']],
    'X': [['a']],
    'R': [['X', 'B']]
}

Gramar2 = {
    'S': [['A','B'], ['C','D'], ['C','B'], ['S','S']],
    'A': [['B','C'], ['a']],
    'B': [['S','C'], ['b']],
    'C': [['D','D'], ['b']],
    'D': [['B','A']]
}

print(es_cnf(Gramar1))
print(es_cnf(Gramar2))
print(cky(Gramar1,['a', 'b', 'a', 'b']))
print(cky(Gramar1,['a', 'b', 'b', 'a']))


True
True
[[{'S', 'X', 'B'}, {'R'}, {'A'}, set()], [set(), {'S', 'B'}, set(), set()], [set(), set(), {'S', 'X', 'B'}, {'R'}], [set(), set(), set(), {'S', 'B'}]]
False
[[{'S', 'X', 'B'}, {'R'}, {'A'}, {'S', 'B'}], [set(), {'S', 'B'}, set(), set()], [set(), set(), {'S', 'B'}, set()], [set(), set(), set(), {'S', 'X', 'B'}]]
True


In [ ]:
# OUTPUT
# Boolean 

In [8]:
# TEST CASES

Gramar1 = {
    'S': [['a'], ['X', 'A'], ['A', 'X'], ['b']],
    'A': [['R', 'B']],
    'B': [['A', 'X'], ['b'], ['a']],
    'X': [['a']],
    'R': [['X', 'B']]
}

Gramar2 = {
    'S': [['A','B'], ['C','D'], ['C','B'], ['S','S']],
    'A': [['B','C'], ['a']],
    'B': [['S','C'], ['b']],
    'C': [['D','D'], ['b']],
    'D': [['B','A']]
}

In [15]:
# EXAMPLE 1 TEST CASE 1
Gramar = Gramar1
Word1 = ['a', 'b', 'a', 'b']

boolean1 = test(Word1, Gramar)
print(f"Word {Word1} can be generated by the grammar: {boolean1}")
#False

# EXAMPLE 2 TEST CASE 1
Word2 = ['a', 'b', 'b', 'a']
boolean2 = test(Word2, Gramar)
print(f"Word {Word2} can be generated by the grammar: {boolean2}")
# True


Grammar is in CNF
Word ['a', 'b', 'a', 'b'] can be generated by the grammar: False
Grammar is in CNF
Word ['a', 'b', 'b', 'a'] can be generated by the grammar: False


In [13]:
# EXAMPLE 2
Gramar = Gramar2
Word = ['a', 'b', 'b', 'a']

boolean = test(Word, Gramar)
print(f"Word {Word} can be generated by the grammar: {boolean}")

Grammar is in CNF
Word ['a', 'b', 'b', 'a'] can be generated by the grammar: False


In [ ]:
# EXTENSION 1
# Transformacion de cualquier gramatica CFG a CNF.

In [ ]:
# EXTENSION 2
# ampliar el algoritmo CKY a probabilıstico.